## 1️⃣ Setup and Imports

In [ ]:
# Import required libraries
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2️⃣ Configuration

In [ ]:
# Configuration
DATASET_DIR = Path('dataset')
OUTPUT_DIR = Path('training_output')
MODELS_DIR = OUTPUT_DIR / 'models'
GRAPHS_DIR = OUTPUT_DIR / 'graphs'
LOGS_DIR = OUTPUT_DIR / 'logs'

# Create directories
for dir_path in [OUTPUT_DIR, MODELS_DIR, GRAPHS_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Model parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
CHANNELS = 3
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# Class names
CLASS_NAMES = [
    'Bract Mosaic Virus',
    'Cordana',
    'Healthy',
    'Panama Disease',
    'Pestalotiopsis',
    'sigatoka'
]

print("✅ Configuration complete")
print(f"📂 Dataset: {DATASET_DIR}")
print(f"📊 Batch size: {BATCH_SIZE}")
print(f"🔢 Epochs: {EPOCHS}")

## 3️⃣ Load Dataset Information

In [ ]:
# Load dataset info
with open(DATASET_DIR / 'dataset_info.json', 'r') as f:
    dataset_info = json.load(f)

# Display dataset statistics
df_classes = pd.DataFrame(dataset_info['classes'])
df_classes = df_classes.sort_values('count', ascending=False)

print(f"📊 Dataset: {dataset_info['dataset_name']}")
print(f"📷 Total Images: {dataset_info['total_images']}")
print(f"🏷️  Classes: {dataset_info['num_classes']}\n")

display(df_classes)

# Visualize class distribution
plt.figure(figsize=(12, 6))
colors = [c['color'] for c in dataset_info['classes']]
plt.bar(df_classes['name'], df_classes['count'], color=colors, alpha=0.7, edgecolor='black')
plt.title('Dataset Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'dataset_distribution.png', dpi=300)
plt.show()

print("⚠️  Note: Class imbalance detected!")
print("   - Will use class weights during training")
print("   - Data augmentation enabled")

## 4️⃣ Create Data Generators

In [ ]:
# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.30  # 70% train, 30% val
)

# Validation data generator (no augmentation)
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.30
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

print(f"✅ Training samples: {train_generator.samples}")
print(f"✅ Validation samples: {val_generator.samples}")
print(f"\n📋 Class indices: {train_generator.class_indices}")

## 5️⃣ Visualize Sample Images

In [ ]:
# Get a batch of images
sample_images, sample_labels = next(train_generator)

# Get class names from indices
class_indices = train_generator.class_indices
class_names_map = {v: k for k, v in class_indices.items()}

# Plot sample images
fig, axes = plt.subplots(4, 4, figsize=(15, 15))
fig.suptitle('Sample Training Images with Augmentation', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        label_idx = np.argmax(sample_labels[i])
        ax.set_title(class_names_map[label_idx], fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'sample_images.png', dpi=300)
plt.show()

## 6️⃣ Compute Class Weights

In [ ]:
# Compute class weights for imbalanced dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)

class_weight_dict = dict(enumerate(class_weights))

print("⚖️  Class Weights:")
for idx, (class_name, weight) in enumerate(zip(train_generator.class_indices.keys(), class_weights)):
    print(f"  {class_name}: {weight:.2f}")

# Visualize class weights
plt.figure(figsize=(10, 6))
plt.bar(train_generator.class_indices.keys(), class_weights, color='skyblue', edgecolor='black')
plt.title('Class Weights (Higher = More Emphasis)', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Weight', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / 'class_weights.png', dpi=300)
plt.show()

## 7️⃣ Build MobileNetV3 Model

In [ ]:
# Load pre-trained MobileNetV3Large
base_model = MobileNetV3Large(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build custom classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(len(CLASS_NAMES), activation='softmax')
])

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top_2_accuracy')]
)

print("✅ Model built successfully!\n")
model.summary()

## 8️⃣ Setup Training Callbacks

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

callbacks = [
    ModelCheckpoint(
        filepath=str(MODELS_DIR / f'best_model_{timestamp}.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

print("✅ Callbacks configured:")
print("   - Model checkpoint (best validation accuracy)")
print("   - Early stopping (patience=10)")
print("   - Learning rate reduction (patience=5)")

## 9️⃣ Train the Model

In [ ]:
print("🚀 Starting training...\n")
print("=" * 80)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

print("\n" + "=" * 80)
print("✅ Training completed!")

## 🔟 Plot Training History (Accuracy & Loss Graphs)

In [ ]:
# Create comprehensive training plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('MobileNetV3 Training Analysis', fontsize=18, fontweight='bold', y=1.00)

# Plot 1: Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2.5, marker='o', markersize=4)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2.5, marker='s', markersize=4)
axes[0, 0].set_title('Model Accuracy', fontsize=14, fontweight='bold', pad=10)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Accuracy', fontsize=12)
axes[0, 0].legend(loc='lower right', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim([0, 1])

# Plot 2: Loss
axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2.5, marker='o', markersize=4)
axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2.5, marker='s', markersize=4)
axes[0, 1].set_title('Model Loss', fontsize=14, fontweight='bold', pad=10)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Loss', fontsize=12)
axes[0, 1].legend(loc='upper right', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Top-2 Accuracy
axes[1, 0].plot(history.history['top_2_accuracy'], label='Train Top-2', linewidth=2.5, marker='o', markersize=4)
axes[1, 0].plot(history.history['val_top_2_accuracy'], label='Val Top-2', linewidth=2.5, marker='s', markersize=4)
axes[1, 0].set_title('Top-2 Accuracy', fontsize=14, fontweight='bold', pad=10)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontsize=12)
axes[1, 0].legend(loc='lower right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 1])

# Plot 4: Learning Rate
lr_history = history.history.get('lr', [LEARNING_RATE] * len(history.history['loss']))
axes[1, 1].plot(lr_history, linewidth=2.5, color='orange', marker='o', markersize=4)
axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold', pad=10)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'training_history_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Training graphs saved to {GRAPHS_DIR / f'training_history_{timestamp}.png'}")

## 1️⃣1️⃣ Generate Confusion Matrix

In [ ]:
# Get predictions on validation set
print("🎯 Evaluating model on validation set...\n")

val_generator.reset()
predictions = model.predict(val_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_generator.classes

# Get class names
class_names = list(val_generator.class_indices.keys())

# Compute confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Plot confusion matrix
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'},
    linewidths=0.5,
    linecolor='gray'
)
plt.title('Confusion Matrix - MobileNetV3', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'confusion_matrix_{timestamp}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Confusion matrix saved to {GRAPHS_DIR / f'confusion_matrix_{timestamp}.png'}")

## 1️⃣2️⃣ Classification Report & Per-Class Accuracy

In [ ]:
# Print classification report
print("📊 Classification Report:\n")
report = classification_report(
    true_classes, 
    predicted_classes, 
    target_names=class_names,
    digits=4
)
print(report)

# Per-class accuracy
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n📈 Per-Class Accuracy:")
print("=" * 50)
accuracy_df = pd.DataFrame({
    'Class': class_names,
    'Accuracy': [f"{acc*100:.2f}%" for acc in class_accuracy],
    'Correct': cm.diagonal(),
    'Total': cm.sum(axis=1)
}).sort_values('Accuracy', ascending=False)

display(accuracy_df)

# Visualize per-class accuracy
plt.figure(figsize=(12, 6))
colors_map = {c['name']: c['color'] for c in dataset_info['classes']}
bar_colors = [colors_map.get(name, '#888888') for name in class_names]
plt.bar(class_names, class_accuracy * 100, color=bar_colors, alpha=0.7, edgecolor='black')
plt.axhline(y=np.mean(class_accuracy) * 100, color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(class_accuracy)*100:.2f}%')
plt.title('Per-Class Accuracy', fontsize=14, fontweight='bold')
plt.xlabel('Disease Class', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim([0, 100])
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(GRAPHS_DIR / f'per_class_accuracy_{timestamp}.png', dpi=300)
plt.show()

## 1️⃣3️⃣ Convert to TensorFlow Lite

In [ ]:
# Convert to TFLite
print("🔄 Converting model to TensorFlow Lite...\n")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save TFLite model
tflite_path = MODELS_DIR / f'spotbleaf_model_{timestamp}.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

# Save labels
labels_path = MODELS_DIR / f'labels_{timestamp}.txt'
with open(labels_path, 'w') as f:
    for class_name in CLASS_NAMES:
        f.write(f"{class_name}\n")

# Get file sizes
keras_size = os.path.getsize(MODELS_DIR / f'best_model_{timestamp}.keras') / (1024 * 1024)
tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)

print(f"✅ Keras model size: {keras_size:.2f} MB")
print(f"✅ TFLite model size: {tflite_size:.2f} MB")
print(f"✅ Compression: {(1 - tflite_size/keras_size)*100:.1f}%\n")
print(f"📁 TFLite model: {tflite_path}")
print(f"📁 Labels file: {labels_path}")

## 1️⃣4️⃣ Final Summary

In [ ]:
# Training summary
summary = {
    'timestamp': timestamp,
    'model': 'MobileNetV3Large',
    'dataset': {
        'total_images': dataset_info['total_images'],
        'train_samples': train_generator.samples,
        'val_samples': val_generator.samples
    },
    'hyperparameters': {
        'batch_size': BATCH_SIZE,
        'epochs': len(history.history['loss']),
        'initial_learning_rate': LEARNING_RATE,
        'image_size': f"{IMG_HEIGHT}x{IMG_WIDTH}"
    },
    'final_metrics': {
        'train_accuracy': float(history.history['accuracy'][-1]),
        'val_accuracy': float(history.history['val_accuracy'][-1]),
        'train_loss': float(history.history['loss'][-1]),
        'val_loss': float(history.history['val_loss'][-1]),
        'best_val_accuracy': float(max(history.history['val_accuracy']))
    },
    'per_class_accuracy': {name: float(acc) for name, acc in zip(class_names, class_accuracy)}
}

# Save summary
with open(LOGS_DIR / f'training_summary_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Display summary
print("=" * 80)
print("🎉 TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"\n📊 Final Results:")
print(f"  Best Validation Accuracy: {summary['final_metrics']['best_val_accuracy']*100:.2f}%")
print(f"  Final Train Accuracy: {summary['final_metrics']['train_accuracy']*100:.2f}%")
print(f"  Final Val Accuracy: {summary['final_metrics']['val_accuracy']*100:.2f}%")
print(f"  Final Train Loss: {summary['final_metrics']['train_loss']:.4f}")
print(f"  Final Val Loss: {summary['final_metrics']['val_loss']:.4f}")

print(f"\n📁 Output Files:")
print(f"  Models: {MODELS_DIR}")
print(f"  Graphs: {GRAPHS_DIR}")
print(f"  Logs: {LOGS_DIR}")

print(f"\n🔄 Next Steps:")
print(f"  1. Copy {tflite_path.name} to ../assets/models/spotbleaf_model_50epochs.tflite")
print(f"  2. Copy {labels_path.name} to ../assets/models/spotbleaf_model_labels.txt")
print(f"  3. Run the Flutter app to test disease detection!")
print("=" * 80)